# AlphaGen US RL Training（本地版）

这个 notebook 是 `colab_us_rl_training.ipynb` 的本地化版本，区别：

- 跑在本地 Python 虚拟环境（默认 `<repo>/.venv`）里，不依赖 Colab/Drive。
- Qlib 美股数据放在 `~/.qlib/qlib_data/us_data`；如果缺失，会通过 `qlib` 自带 CLI 下载到该路径。
- 训练产物保存在仓库的 `out/results/<run_name>/`，与 Colab 同步过来的 `notebooks/runs/` 分开。
- 默认超参用我们目前的 champion 组合（pool=8、LR=3e-5、ent=0.01、clip=0.2、20K 步、每 rollout 存档），可直接重现最近的实验。

**首次使用先在仓库根目录建好 venv**：

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name alphagen-venv  # 让 Jupyter 看到这个 kernel
jupyter notebook notebooks/local_us_rl_training.ipynb
```

之后在 Jupyter 顶栏选择 kernel `alphagen-venv` 即可。


In [ ]:
# 作用：确认 notebook 跑在仓库根目录 + 本地 venv，设好 Apple Silicon 友好的环境变量，并列出关键依赖版本。
import os
# Must set BEFORE importing torch: lets MPS silently fall back to CPU for
# unsupported ops (alphagen 的 Med/Rank 等会用到 argsort/median 在 MPS 上未必稳定)
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'alphagen').is_dir() or not (REPO_ROOT / 'requirements.txt').exists():
    raise RuntimeError(
        f'Please launch Jupyter from the alphagen repo root. Current cwd: {REPO_ROOT}'
    )

VENV_ROOT = REPO_ROOT / '.venv'
in_venv = sys.prefix != sys.base_prefix
print('Python executable :', sys.executable)
print('Python prefix     :', sys.prefix)
print('Repo root         :', REPO_ROOT)
print('Expected venv dir :', VENV_ROOT, '(exists)' if VENV_ROOT.exists() else '(MISSING)')
print('Running in a venv :', in_venv)
print('MPS fallback env  :', os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'))

if not in_venv:
    print()
    print('WARNING: this kernel is not inside a virtual environment.')
    print('Set up the project venv from the repo root:')
    print('  python3 -m venv .venv && source .venv/bin/activate')
    print('  python -m pip install --upgrade pip')
    print('  python -m pip install -r requirements.txt')
    print('  python -m ipykernel install --user --name alphagen-venv')
    print('Then switch the Jupyter kernel to alphagen-venv and rerun this cell.')

import importlib.metadata as _md
required = [
    'numpy', 'pandas', 'scikit-learn', 'torch',
    'stable-baselines3', 'sb3-contrib', 'pyqlib', 'fire', 'protobuf',
]
print()
print('Key package versions:')
for pkg in required:
    try:
        print(f'  {pkg:20s} {_md.version(pkg)}')
    except _md.PackageNotFoundError:
        print(f'  {pkg:20s} MISSING -- run pip install -r requirements.txt inside the venv')

import torch
import platform

# Apple Silicon (M1/M2/M3): cap PyTorch threads to perf-core count.
# Using all logical cores (which includes efficiency cores) often slows things
# down on M-series because P/E cores share memory bandwidth.
is_apple_silicon = (platform.system() == 'Darwin') and (platform.machine() == 'arm64')
if is_apple_silicon:
    perf_threads = 4  # M1/M2 base: 4 P-cores; raise to 6/8 on M2/M3 Pro/Max
    torch.set_num_threads(perf_threads)
    try:
        torch.set_num_interop_threads(2)
    except RuntimeError:
        pass  # already set during this process
    print()
    print(f'Apple Silicon detected -> torch threads capped to {perf_threads} (perf cores)')

print()
print('Torch CUDA available :', torch.cuda.is_available())
print('Torch MPS available  :', getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available())
print('Torch num_threads    :', torch.get_num_threads())

# Make repo importable for the rest of the notebook (jupyter may default to notebook dir).
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd set to repo root.')


In [ ]:
# 作用：集中定义数据路径、时间切分和训练超参（与 colab notebook 的 config cell 平行，但全部本地路径）。
from pathlib import Path

LOCAL_QLIB_DIR = Path('~/.qlib/qlib_data/us_data').expanduser()
SAVE_ROOT = REPO_ROOT / 'out' / 'results'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

MAX_BACKTRACK_DAYS = 100
MAX_FUTURE_DAYS = 30
AUTO_TRAIN_START = '2010-01-01'
PRE_PANDEMIC_TEST_END = '2020-01-31'
VALID_TRADING_DAYS = 252
TEST_TRADING_DAYS = 252

SEGMENTS = None
CALENDAR_INFO = None

# 'Champion' defaults reproducing the latest Colab pool=8 / LR=3e-5 run.
SEEDS = (0, 1, 2)
POOL_CAPACITY = 8
TRAINING_STEPS = 20_000
PPO_N_STEPS = 2048
BATCH_SIZE = 128
PRINT_EXPR = False
LEARNING_RATE = 3e-5
LR_SCHEDULE = 'constant'  # 'constant' or 'linear' (decay LEARNING_RATE -> 0 over TRAINING_STEPS)
ENT_COEF = 0.01           # raise (0.02~0.05) to keep exploration alive
CLIP_RANGE = 0.2          # lower (0.1) for more conservative updates
CHECKPOINT_EVERY_N_ROLLOUTS = 1
MODEL_CHECKPOINT_START_FRACTION = 0.0
CHECKPOINT_SELECTION_METRIC = 'valid_rank_icir'
US_DELTA_TIMES = (1, 5, 10, 20, 40, 60)

# 首次本地跑通流程时把 SMOKE_TEST 设为 True，可以在 5-10 分钟内拿到一个完整 run。
# 期间会临时压低训练规模；想跑真正实验时记得改回 False。
SMOKE_TEST = False
if SMOKE_TEST:
    SEEDS = (0,)
    TRAINING_STEPS = 4096
    PPO_N_STEPS = 1024            # MPS 上小一些的 rollout 更友好
    print('*** SMOKE_TEST = True ***')
    print(f'    overriding SEEDS={SEEDS}, TRAINING_STEPS={TRAINING_STEPS}, PPO_N_STEPS={PPO_N_STEPS}')

PREFERRED_INSTRUMENTS = ('sp500', 'SP500', 'all', 'ALL')

print('Local Qlib data dir :', LOCAL_QLIB_DIR, '(exists)' if LOCAL_QLIB_DIR.exists() else '(MISSING - will download)')
print('Output root         :', SAVE_ROOT)
print('Seeds               :', SEEDS)
print('Pool capacity       :', POOL_CAPACITY)
print('Training steps      :', TRAINING_STEPS, '| LR:', LEARNING_RATE, '| schedule:', LR_SCHEDULE)
print('PPO ent_coef        :', ENT_COEF, '| clip_range:', CLIP_RANGE)
print('US delta windows    :', US_DELTA_TIMES)


In [ ]:
# 作用：确保本地有 US Qlib 数据，并基于 calendar 自动算出疫情前的 train/valid/test 切分。
import subprocess
import sys
from pathlib import Path
import pandas as pd


def download_qlib_us_data(target_dir: Path):
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(
            [sys.executable, '-m', 'qlib.cli.data', 'qlib_data',
             '--target_dir', str(target_dir), '--region', 'us'],
            check=True,
        )
    except subprocess.CalledProcessError:
        print('qlib.cli.data CLI failed, falling back to qlib.tests.data.GetData().qlib_data(...)')
        from qlib.tests.data import GetData
        GetData().qlib_data(target_dir=str(target_dir), region='us', exists_skip=True)


def build_segments_from_calendar(calendar_path: Path, split_end: str):
    calendar = pd.read_csv(calendar_path, header=None, names=['date'])
    calendar['date'] = pd.to_datetime(calendar['date'])
    usable_dates = pd.Index(
        calendar['date'].iloc[MAX_BACKTRACK_DAYS : len(calendar) - MAX_FUTURE_DAYS]
    )
    split_end_ts = pd.Timestamp(split_end)
    usable_dates = usable_dates[usable_dates <= split_end_ts]
    required_days = VALID_TRADING_DAYS + TEST_TRADING_DAYS + 1
    if len(usable_dates) < required_days:
        raise ValueError(
            f'Qlib US calendar too short for the requested split: {len(usable_dates)} usable rows '
            f'before {split_end}, need at least {required_days}.'
        )
    train_start_idx = int(usable_dates.searchsorted(pd.Timestamp(AUTO_TRAIN_START)))
    last_train_end_idx = len(usable_dates) - VALID_TRADING_DAYS - TEST_TRADING_DAYS - 1
    if train_start_idx > last_train_end_idx:
        raise ValueError('Not enough usable days to build the pre-pandemic train/valid/test splits.')
    valid_start_idx = last_train_end_idx + 1
    valid_end_idx = valid_start_idx + VALID_TRADING_DAYS - 1
    test_start_idx = valid_end_idx + 1

    def fmt(ts):
        return pd.Timestamp(ts).strftime('%Y-%m-%d')

    segments = {
        'train': (fmt(usable_dates[train_start_idx]), fmt(usable_dates[last_train_end_idx])),
        'valid': (fmt(usable_dates[valid_start_idx]), fmt(usable_dates[valid_end_idx])),
        'test':  (fmt(usable_dates[test_start_idx]),  fmt(usable_dates[-1])),
    }
    info = {
        'calendar_start': fmt(calendar['date'].iloc[0]),
        'calendar_end':   fmt(calendar['date'].iloc[-1]),
        'usable_start':   fmt(usable_dates[0]),
        'usable_end':     fmt(usable_dates[-1]),
        'calendar_rows':  int(len(calendar)),
        'usable_rows':    int(len(usable_dates)),
        'requested_split_end': split_end,
        'selected_split_end':  fmt(usable_dates[-1]),
    }
    return segments, info


calendar_path = LOCAL_QLIB_DIR / 'calendars' / 'day.txt'
if not calendar_path.exists():
    print(f'Qlib calendar not found at {calendar_path}, downloading US data...')
    download_qlib_us_data(LOCAL_QLIB_DIR)
    if not calendar_path.exists():
        raise FileNotFoundError(
            f'After download, {calendar_path} still missing. 检查 pyqlib 是否还能访问公开数据源。'
        )

SEGMENTS, CALENDAR_INFO = build_segments_from_calendar(calendar_path, PRE_PANDEMIC_TEST_END)

instrument_files = sorted((LOCAL_QLIB_DIR / 'instruments').glob('*.txt'))
available_instruments = [path.stem for path in instrument_files]
SELECTED_INSTRUMENT = next((n for n in PREFERRED_INSTRUMENTS if n in available_instruments), None)
if SELECTED_INSTRUMENT is None:
    raise ValueError(
        f'No preferred US instrument set found. Available: {available_instruments[:20]}. '
        '改 PREFERRED_INSTRUMENTS 或下载/构造对应 instruments 文件。'
    )

print('Calendar coverage    :', CALENDAR_INFO['calendar_start'], '->', CALENDAR_INFO['calendar_end'])
print('Usable range         :', CALENDAR_INFO['usable_start'], '->', CALENDAR_INFO['usable_end'])
print('Selected instrument  :', SELECTED_INSTRUMENT)
print('Auto-selected splits :', SEGMENTS)


In [ ]:
# 作用：把 RL 环境补丁成美股五特征版本，并同步更新动作空间和窗口设置。
import numpy as np
import alphagen.config as config_module
import alphagen.rl.env.wrapper as env_wrapper_module
import alphagen_qlib.stock_data as stock_data_module
import scripts.rl as rl_module
from alphagen.data.tokens import (
    ConstantToken, DeltaTimeToken, ExpressionToken, FeatureToken,
    OperatorToken, SequenceIndicatorToken, SequenceIndicatorType,
)
from alphagen_qlib.stock_data import FeatureType

US_FEATURES = (
    FeatureType.OPEN, FeatureType.CLOSE, FeatureType.HIGH,
    FeatureType.LOW, FeatureType.VOLUME,
)
US_FEATURE_NAMES = tuple(f.name.lower() for f in US_FEATURES)

config_module.DELTA_TIMES = list(US_DELTA_TIMES)
env_wrapper_module.DELTA_TIMES = list(US_DELTA_TIMES)

OriginalStockData = stock_data_module.StockData

class USStockData(OriginalStockData):
    def __init__(self, *args, features=None, **kwargs):
        if features is None:
            features = list(US_FEATURES)
        super().__init__(*args, features=features, **kwargs)

stock_data_module.StockData = USStockData
rl_module.StockData = USStockData

env_wrapper_module.SIZE_FEATURE = len(US_FEATURES)
env_wrapper_module.SIZE_DELTA_TIME = len(env_wrapper_module.DELTA_TIMES)
env_wrapper_module.SIZE_ACTION = (
    env_wrapper_module.SIZE_OP
    + env_wrapper_module.SIZE_FEATURE
    + env_wrapper_module.SIZE_DELTA_TIME
    + env_wrapper_module.SIZE_CONSTANT
    + env_wrapper_module.SIZE_SEP
)


def patched_action_masks(self):
    res = np.zeros(self.size_action, dtype=bool)
    valid = self.env.valid_action_types()
    offset = 0
    for i in range(offset, offset + env_wrapper_module.SIZE_OP):
        if valid['op'][env_wrapper_module.OPERATORS[i - offset].category_type()]:
            res[i] = True
    offset += env_wrapper_module.SIZE_OP
    if valid['select'][1]:
        res[offset:offset + env_wrapper_module.SIZE_FEATURE] = True
    offset += env_wrapper_module.SIZE_FEATURE
    if valid['select'][2]:
        res[offset:offset + env_wrapper_module.SIZE_CONSTANT] = True
    offset += env_wrapper_module.SIZE_CONSTANT
    if valid['select'][3]:
        res[offset:offset + env_wrapper_module.SIZE_DELTA_TIME] = True
    offset += env_wrapper_module.SIZE_DELTA_TIME
    if valid['select'][1]:
        res[offset:offset + len(self.subexprs)] = True
    offset += len(self.subexprs)
    if valid['select'][4]:
        res[offset] = True
    return res


def patched_action_to_token(self, action: int):
    if action < 0:
        raise ValueError
    if action < env_wrapper_module.SIZE_OP:
        return OperatorToken(env_wrapper_module.OPERATORS[action])
    action -= env_wrapper_module.SIZE_OP
    if action < env_wrapper_module.SIZE_FEATURE:
        return FeatureToken(US_FEATURES[action])
    action -= env_wrapper_module.SIZE_FEATURE
    if action < env_wrapper_module.SIZE_CONSTANT:
        return ConstantToken(env_wrapper_module.CONSTANTS[action])
    action -= env_wrapper_module.SIZE_CONSTANT
    if action < env_wrapper_module.SIZE_DELTA_TIME:
        return DeltaTimeToken(env_wrapper_module.DELTA_TIMES[action])
    action -= env_wrapper_module.SIZE_DELTA_TIME
    if action < len(self.subexprs):
        return ExpressionToken(self.subexprs[action])
    action -= len(self.subexprs)
    if action == 0:
        return SequenceIndicatorToken(SequenceIndicatorType.SEP)
    raise AssertionError('Invalid action index')


env_wrapper_module.AlphaEnvWrapper.action_masks = patched_action_masks
env_wrapper_module.AlphaEnvWrapper.action_to_token = patched_action_to_token

print('US notebook patch active:', US_FEATURE_NAMES)
print('Action space size       :', env_wrapper_module.SIZE_ACTION)


In [ ]:
# 作用：按多个 seed 跑 PPO 训练，每个 rollout 落一份 *_pool.json，便于 valid 选档。
import torch
from scripts.rl import run_single_experiment, status

if SEGMENTS is None:
    raise RuntimeError('SEGMENTS not initialized; run the data-prep cell first.')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

SEGMENT_ORDER = ('train', 'valid', 'test')
SEGMENT_TUPLES = tuple(SEGMENTS[name] for name in SEGMENT_ORDER)
MODEL_CHECKPOINT_START_STEP = max(0, int(TRAINING_STEPS * MODEL_CHECKPOINT_START_FRACTION))
RUN_DIRS = []

print('Training device     :', DEVICE)
print('Training segments   :', dict(zip(SEGMENT_ORDER, SEGMENT_TUPLES)))
print('Checkpoint cadence  :', CHECKPOINT_EVERY_N_ROLLOUTS, 'rollout(s)')
print('Model save start    :', MODEL_CHECKPOINT_START_STEP, 'steps')
print('LR / schedule       :', LEARNING_RATE, '/', LR_SCHEDULE)
print('ent_coef / clip     :', ENT_COEF, '/', CLIP_RANGE)
print()

for seed in SEEDS:
    run_dir = Path(
        run_single_experiment(
            seed=seed,
            instruments=SELECTED_INSTRUMENT,
            pool_capacity=POOL_CAPACITY,
            steps=TRAINING_STEPS,
            qlib_data_path=str(LOCAL_QLIB_DIR),
            qlib_region='us',
            device=DEVICE,
            segments=SEGMENT_TUPLES,
            ppo_n_steps=PPO_N_STEPS,
            batch_size=BATCH_SIZE,
            print_expr=PRINT_EXPR,
            checkpoint_every_n_rollouts=CHECKPOINT_EVERY_N_ROLLOUTS,
            model_checkpoint_start_step=MODEL_CHECKPOINT_START_STEP,
            learning_rate=LEARNING_RATE,
            lr_schedule=LR_SCHEDULE,
            ent_coef=ENT_COEF,
            clip_range=CLIP_RANGE,
        )
    )
    RUN_DIRS.append(run_dir)
    print('Completed seed', seed, '->', run_dir)
    status(str(run_dir))

print()
print('All run dirs:')
for d in RUN_DIRS:
    print(' ', d)


In [ ]:
# 作用：遍历每个 run 的全部 checkpoint，按 valid_rank_icir 挑出最佳 checkpoint，并把指标落地到 CSV。
import json
import pandas as pd

from alphagen.data.expression import Feature, Ref
from alphagen.models.linear_alpha_pool import MseAlphaPool
from alphagen_qlib.calculator import QLibStockDataCalculator
from alphagen_qlib.stock_data import FeatureType, StockData, initialize_qlib
from alphagen_qlib.utils import load_alpha_pool_by_path

if not RUN_DIRS:
    raise RuntimeError('RUN_DIRS is empty; run the training cell first.')

initialize_qlib(str(LOCAL_QLIB_DIR), region='us')
close = Feature(FeatureType.CLOSE)
target = Ref(close, -20) / close - 1

split_calculators = {}
for split_name, (start_time, end_time) in SEGMENTS.items():
    data = StockData(
        instrument=SELECTED_INSTRUMENT,
        start_time=start_time,
        end_time=end_time,
        device=DEVICE,
    )
    split_calculators[split_name] = QLibStockDataCalculator(data, target)


def evaluate_checkpoint(pool_path):
    exprs, weights = load_alpha_pool_by_path(str(pool_path))
    rows = []
    for split_name, (start_time, end_time) in SEGMENTS.items():
        calculator = split_calculators[split_name]
        data = calculator.data
        pool = MseAlphaPool(
            capacity=max(POOL_CAPACITY, len(exprs)),
            calculator=calculator,
            ic_lower_bound=None,
            l1_alpha=5e-3,
            device=DEVICE,
        )
        pool.force_load_exprs(exprs, weights=weights)
        ic, rank_ic = pool.test_ensemble(calculator)
        ic_mean, icir, rank_ic_mean, rank_icir = calculator.calc_pool_all_ret_with_ir(exprs, weights)
        rows.append({
            'split': split_name,
            'start_time': start_time,
            'end_time': end_time,
            'n_days': int(data.n_days),
            'n_stocks': int(data.n_stocks),
            'ic': float(ic),
            'rank_ic': float(rank_ic),
            'icir': float(icir),
            'rank_icir': float(rank_icir),
            'pool_size': len(exprs),
            'checkpoint': pool_path.name,
        })
    return pd.DataFrame(rows)


seed_summaries = []
best_metrics_frames = []

for run_dir in RUN_DIRS:
    checkpoint_paths = sorted(
        run_dir.glob('*_steps_pool.json'),
        key=lambda path: int(path.name.split('_', 1)[0]),
    )
    if not checkpoint_paths:
        raise FileNotFoundError(f'No *_steps_pool.json under {run_dir}')

    run_config = json.loads((run_dir / 'run_config.json').read_text(encoding='utf-8'))
    seed = int(run_config['seed'])
    checkpoint_metrics = []
    checkpoint_summary_rows = []
    best_metrics_by_checkpoint = {}

    for pool_path in checkpoint_paths:
        metrics_df = evaluate_checkpoint(pool_path)
        metrics_df['seed'] = seed
        metrics_df['run_dir'] = run_dir.name
        checkpoint_metrics.append(metrics_df)
        best_metrics_by_checkpoint[pool_path.name] = metrics_df
        valid_row = metrics_df.loc[metrics_df['split'] == 'valid'].iloc[0]
        checkpoint_summary_rows.append({
            'seed': seed,
            'run_dir': run_dir.name,
            'checkpoint': pool_path.name,
            'step': int(pool_path.name.split('_', 1)[0]),
            'valid_ic': float(valid_row['ic']),
            'valid_rank_ic': float(valid_row['rank_ic']),
            'valid_icir': float(valid_row['icir']),
            'valid_rank_icir': float(valid_row['rank_icir']),
        })

    checkpoint_metrics_df = pd.concat(checkpoint_metrics, ignore_index=True)
    checkpoint_summary_df = pd.DataFrame(checkpoint_summary_rows).sort_values(
        by=['valid_rank_icir', 'valid_rank_ic', 'valid_ic'],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    best_checkpoint = checkpoint_summary_df.iloc[0]['checkpoint']
    best_metrics_df = best_metrics_by_checkpoint[best_checkpoint].copy()
    best_metrics_df['selected_by'] = CHECKPOINT_SELECTION_METRIC
    best_metrics_df['is_best_checkpoint'] = True

    checkpoint_metrics_df.to_csv(run_dir / 'checkpoint_metrics.csv', index=False)
    checkpoint_summary_df.to_csv(run_dir / 'checkpoint_selection.csv', index=False)
    best_metrics_df.to_csv(run_dir / 'best_segment_metrics.csv', index=False)
    (run_dir / 'local_config.json').write_text(
        json.dumps(
            {
                'seed': seed,
                'seeds': list(SEEDS),
                'pool_capacity': POOL_CAPACITY,
                'training_steps': TRAINING_STEPS,
                'ppo_n_steps': PPO_N_STEPS,
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE,
                'lr_schedule': LR_SCHEDULE,
                'ent_coef': ENT_COEF,
                'clip_range': CLIP_RANGE,
                'instrument': SELECTED_INSTRUMENT,
                'qlib_data_path': str(LOCAL_QLIB_DIR),
                'segments': SEGMENTS,
                'calendar_info': CALENDAR_INFO,
                'feature_names': US_FEATURE_NAMES,
                'delta_times': list(US_DELTA_TIMES),
                'device': str(DEVICE),
                'checkpoint_every_n_rollouts': CHECKPOINT_EVERY_N_ROLLOUTS,
                'model_checkpoint_start_fraction': MODEL_CHECKPOINT_START_FRACTION,
                'model_checkpoint_start_step': MODEL_CHECKPOINT_START_STEP,
                'checkpoint_selection_metric': CHECKPOINT_SELECTION_METRIC,
                'best_checkpoint': best_checkpoint,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )

    best_metrics_frames.append(best_metrics_df)
    summary = checkpoint_summary_df.iloc[0].to_dict()
    summary['best_checkpoint'] = best_checkpoint
    seed_summaries.append(summary)

    print(f'Seed {seed} best checkpoint: {best_checkpoint}')
    print(best_metrics_df)
    print()

best_metrics_all_df = pd.concat(best_metrics_frames, ignore_index=True)
seed_summary_df = pd.DataFrame(seed_summaries)
aggregate_metrics_df = (
    best_metrics_all_df.groupby('split', as_index=False)
    .agg(
        seeds=('seed', 'nunique'),
        ic_mean=('ic', 'mean'),
        ic_std=('ic', 'std'),
        rank_ic_mean=('rank_ic', 'mean'),
        rank_ic_std=('rank_ic', 'std'),
        icir_mean=('icir', 'mean'),
        icir_std=('icir', 'std'),
        rank_icir_mean=('rank_icir', 'mean'),
        rank_icir_std=('rank_icir', 'std'),
        n_days=('n_days', 'first'),
        n_stocks_mean=('n_stocks', 'mean'),
    )
)

aggregate_dir = SAVE_ROOT / 'aggregate'
aggregate_dir.mkdir(parents=True, exist_ok=True)
seed_summary_df.to_csv(aggregate_dir / 'seed_summary.csv', index=False)
aggregate_metrics_df.to_csv(aggregate_dir / 'best_checkpoint_metrics.csv', index=False)
best_metrics_all_df.to_csv(aggregate_dir / 'best_segment_metrics_all.csv', index=False)

print('Per-seed checkpoint summary:')
print(seed_summary_df)
print()
print('Multi-seed aggregate metrics (best checkpoint per seed):')
print(aggregate_metrics_df)
print()
print('Aggregate dir:', aggregate_dir)


## 调参建议（与 Colab notebook 一致，方便对比）

- 默认 `valid/test` 都限制在 `2020-01-31` 之前，避免疫情冲击混进验证/测试。
- 默认训练规模：`TRAINING_STEPS = 20_000`、`PPO_N_STEPS = 2048`、`CHECKPOINT_EVERY_N_ROLLOUTS = 1`，每个 rollout 落一份 `*_pool.json`；`MODEL_CHECKPOINT_START_FRACTION = 0.0` 表示每个 checkpoint 都会保存 `.zip` 模型权重。
- `LEARNING_RATE` 默认 `3e-5`（远低于 SB3 default 3e-4），如果观察到训练前几个 rollout 就过拟合可改用 `LR_SCHEDULE='linear'` + 较大初始 LR。
- `ENT_COEF` 默认 `0.01`；提到 `0.03~0.05` 可鼓励 policy 提出更多样化候选；过高会让无效 token 增多。
- `CLIP_RANGE` 默认 `0.2`；降到 `0.1` 让 PPO 更新更保守，遇到训练曲线锯齿震荡时可考虑——但实测和 LR 同时压低会让 `pool_eval_cnt` 暴跌，单变量改动更安全。
- `POOL_CAPACITY = 8` 比 10 更稳；如果还想压低过拟合容量可试 5/6。
- 评估阶段会按 `valid_rank_icir` 选 best checkpoint，写出 `checkpoint_metrics.csv` / `checkpoint_selection.csv` / `best_segment_metrics.csv` 到每个 run 目录，以及聚合表 `out/results/aggregate/*.csv`。
- 没有 Google Drive 同步：所有产物都在 `out/results/<run_name>/`。需要保留实验请自己复制走或纳入 git。
- **MacBook (Apple Silicon)**：setup cell 自动设 `PYTORCH_ENABLE_MPS_FALLBACK=1` 并把 PyTorch 线程数限制到 4（M2 性能核数）。 训练 cell 会自动选 `mps` 设备。M2 上跑一个 seed × 20K 步大约 20-40 分钟，比 Colab T4 慢 3-5×；做大量超参扫描仍建议用 Colab。
- **首次本地跑通**：把 config cell 里 `SMOKE_TEST = True`，会临时把 `SEEDS=(0,)`、`TRAINING_STEPS=4096`、`PPO_N_STEPS=1024`，整条流水线 5-10 分钟内能跑完，方便验证数据/补丁/评估都没问题。
